# PlaceMux Growth Experimentation — Exploration Notebook

Quick, interactive walkthrough of the same engines used by `app.py`. Useful for ad-hoc analysis and for sanity-checking a new experiment before it hits the dashboard.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import database
from experiment_engine import run_full_readout, _get_metric_arrays
from recommendation_engine import make_recommendation
import validation as validation_module

## 1. What experiments do we have?

In [ ]:
database.list_experiments()

## 2. Full readout for one experiment

In [ ]:
readout = run_full_readout('exp_1001')
readout.primary_result.to_dict()

In [ ]:
readout.srm.to_dict()

In [ ]:
pd.DataFrame([g.to_dict() for g in readout.guardrail_results])

## 3. Ship / no-ship recommendation

In [ ]:
rec = make_recommendation(readout)
print(rec.decision)
print(rec.headline)
for line in rec.reasoning:
    print('-', line)

## 4. Raw per-user distributions behind the test

In [ ]:
control, treatment = _get_metric_arrays('exp_1001', 'application_conversion')
print('control n=', len(control), 'mean=', control.mean())
print('treatment n=', len(treatment), 'mean=', treatment.mean())

## 5. Sanity-check across the whole portfolio

In [ ]:
rows = []
for exp_id in database.list_experiments()['experiment_id']:
    r = run_full_readout(exp_id)
    rec = make_recommendation(r)
    rows.append({
        'experiment_id': exp_id,
        'primary_metric': r.primary_result.metric,
        'relative_lift_pct': r.primary_result.relative_diff_pct,
        'p_value': r.primary_result.p_value,
        'srm_flagged': r.srm.srm_detected,
        'guardrail_regressions': sum(g.is_regression for g in r.guardrail_results),
        'decision': rec.decision,
    })
pd.DataFrame(rows)

## 6. Data validation spot-check

In [ ]:
reports = validation_module.run_all_validations()
pd.DataFrame([
    {'table': t, 'rows': r.row_count, 'missing_cols': len(r.missing_value_counts),
     'dup_pk_rows': r.duplicate_primary_keys, 'passed': r.passed}
    for t, r in reports.items()
])